In [39]:
%load_ext autoreload
%autoreload 2

from collections import Counter
import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# from helpers import plotLightCurveFromDF, sampleRandomKIC, saveLightCurveFromDF
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
df = pd.read_parquet("/Users/tomlinn/Downloads/MERGED_LCS.parquet")
df


,time,flux,Class
KIC,,,
757450,"[131.51271468758932, 131.5331494016791, 131.55...","[1.0392150925008297, 1.0383358659303283, 1.038...",CONFIRMED
892772,"[352.39657371611975, 352.43743948176416, 352.4...","[1.019566672226177, 1.0188446568700726, 1.0189...",FALSE POSITIVE
1025986,"[120.53931629149156, 120.55975094284076, 120.5...","[1.0488531548750866, 1.0523113529281902, 1.051...",CANDIDATE
1026032,"[131.5127135392686, 131.53314824846893, 131.55...","[1.0509398498643578, 1.0503349421979056, 1.050...",FALSE POSITIVE
1026957,"[120.53930025400041, 120.55973490802717, 120.5...","[1.0599377280481261, 1.0591694377327983, 1.058...",CONFIRMED
...,...,...,...
202140012,"[1940.0083829938958, 1940.0288146129678, 1940....","[0.9961692094802856, 0.9965322613716125, 0.996...",VARIABLE STAR
202140013,"[1940.0083988461702, 1940.0288304660571, 1940....","[0.9897362589836121, 0.9907680153846741, 0.991...",VARIABLE STAR
202140059,"[1940.0088655664877, 1940.0292972054667, 1940....","[1.0229097604751587, 1.014359951019287, 1.0042...",FALSE POSITIVE


In [4]:

df = df.reset_index()

In [5]:
renameMap = {
    'time': 'Time',
    'flux': 'Flux'
}

df = df.rename(columns=renameMap)
df

,KIC,Time,Flux,Class
0,757450,"[131.51271468758932, 131.5331494016791, 131.55...","[1.0392150925008297, 1.0383358659303283, 1.038...",CONFIRMED
1,892772,"[352.39657371611975, 352.43743948176416, 352.4...","[1.019566672226177, 1.0188446568700726, 1.0189...",FALSE POSITIVE
2,1025986,"[120.53931629149156, 120.55975094284076, 120.5...","[1.0488531548750866, 1.0523113529281902, 1.051...",CANDIDATE
3,1026032,"[131.5127135392686, 131.53314824846893, 131.55...","[1.0509398498643578, 1.0503349421979056, 1.050...",FALSE POSITIVE
4,1026957,"[120.53930025400041, 120.55973490802717, 120.5...","[1.0599377280481261, 1.0591694377327983, 1.058...",CONFIRMED
...,...,...,...,...
18877,202140012,"[1940.0083829938958, 1940.0288146129678, 1940....","[0.9961692094802856, 0.9965322613716125, 0.996...",VARIABLE STAR
18878,202140013,"[1940.0083988461702, 1940.0288304660571, 1940....","[0.9897362589836121, 0.9907680153846741, 0.991...",VARIABLE STAR
18879,202140059,"[1940.0088655664877, 1940.0292972054667, 1940....","[1.0229097604751587, 1.014359951019287, 1.0042...",FALSE POSITIVE
18880,202140094,"[1940.0088592208049, 1940.029290861763, 1940.0...","[1.008090615272522, 1.0085946321487427, 1.0099...",FALSE POSITIVE


In [6]:
def chooseLabel(labels):
    labels = list(labels)
    nonFp = [lab for lab in labels if lab != "FALSE POSITIVE"]

    if len(nonFp) > 0:
        # If there are multiple non-FP labels, take the most common one
        return pd.Series(nonFp).mode().iloc[0]

    # If all labels are FALSE POSITIVE, keep one of them
    return pd.Series(labels).mode().iloc[0]

In [7]:
df = (
    df.groupby("KIC", as_index=False)
        .agg({
            "Time": "first",
            "Flux": "first",
            "Class": chooseLabel
        })
)

In [8]:
df['Class'].value_counts()

Class
FALSE POSITIVE           7522
ECLIPSING BINARY STAR    3049
VARIABLE STAR            2980
CONFIRMED                1972
CANDIDATE                1637
Name: count, dtype: int64

In [9]:
df.duplicated(subset="KIC").sum()

np.int64(0)

In [10]:
df = df[df['Class'] != 'CANDIDATE'].copy()

In [11]:
def encodeLabels(df):
    """
    Function to encode the labels in the 'Class' feature of a given dataframe df.
    """
    le = LabelEncoder()
    y = le.fit_transform(df["Class"].values)
    df = df.copy()
    df["label"] = y

    return df, le

In [12]:
def normalizeFlux(flux):
    """
    What it does:
        converts flux to a NumPy array
        removes NaN / inf values
        divides by the median flux
        shifts the baseline so the typical value is around zero
    
    Why this matters:
        Different stars can have very different absolute flux scales.
        If you do not normalize, the model may learn scale differences instead of shape differences.

        For a TCN, the important thing is usually the pattern over time, not the raw units.
    """
    flux = np.asarray(flux, dtype=np.float32)
    flux = flux[np.isfinite(flux)]
    if len(flux) == 0:
        raise ValueError("Empty/invalid flux array.")

    med = np.median(flux)
    if med == 0 or not np.isfinite(med):
        med = 1.0
    flux = flux / med - 1.0
    return flux

In [13]:
def resampleToFixedLength(time, flux, seq_len=2000):
    """
    What it does:
        removes invalid values
        sorts the data by time
        removes duplicate timestamps
        interpolates the light curve onto a fixed grid of seq_len points

    Why this matters:
        A TCN needs input tensors of the same shape.

    Your light curves likely have:
        different lengths
        irregular cadences
        gaps

    This step turns every light curve into the same length, so the model can train in batches.
    """
    time = np.asarray(time, dtype=np.float32)
    flux = np.asarray(flux, dtype=np.float32)

    mask = np.isfinite(time) & np.isfinite(flux)
    time = time[mask]
    flux = flux[mask]

    if len(time) < 2:
        raise ValueError("Not enough valid points to resample.")

    order = np.argsort(time)
    time = time[order]
    flux = flux[order]

    # Remove duplicate time stamps
    uniq_time, uniq_idx = np.unique(time, return_index=True)
    time = uniq_time
    flux = flux[uniq_idx]

    if len(time) < 2:
        raise ValueError("Not enough unique time points.")

    t_new = np.linspace(time.min(), time.max(), seq_len)
    y_new = np.interp(t_new, time, flux).astype(np.float32)

    return y_new

In [14]:
def preprocessLightCurve(time, flux, seq_len=2000, clip_sigma=5.0):
    time = np.asarray(time, dtype=np.float32)
    flux = np.asarray(flux, dtype=np.float32)

    # keep time and flux aligned
    mask = np.isfinite(time) & np.isfinite(flux)
    time = time[mask]
    flux = flux[mask]

    if len(time) < 2:
        raise ValueError("Not enough valid points.")

    order = np.argsort(time)
    time = time[order]
    flux = flux[order]

    uniq_time, uniq_idx = np.unique(time, return_index=True)
    time = uniq_time
    flux = flux[uniq_idx]

    if len(time) < 2:
        raise ValueError("Not enough unique time points.")

    med = np.median(flux)
    if med == 0 or not np.isfinite(med):
        med = 1.0
    flux = flux / med - 1.0

    mu = np.mean(flux)
    sd = np.std(flux)
    if sd > 0 and np.isfinite(sd):
        flux = np.clip(flux, mu - clip_sigma * sd, mu + clip_sigma * sd)

    t_new = np.linspace(time.min(), time.max(), seq_len)
    x = np.interp(t_new, time, flux).astype(np.float32)

    x = x - np.mean(x)
    std = np.std(x)
    if std > 0 and np.isfinite(std):
        x = x / std

    return x.astype(np.float32)

In [15]:
class LightCurveDataset(Dataset):
    def __init__(self, df, seq_len=2000):
        self.df = df.reset_index(drop=True)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = preprocessLightCurve(row["Time"], row["Flux"], seq_len=self.seq_len)

        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)  # (1, T)
        y = torch.tensor(row["label"], dtype=torch.long)
        return x, y

In [16]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)

        self.residual = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.dropout(x)
        return F.relu(x + res)


class TCNClassifier(nn.Module):
    def __init__(self, num_classes, in_ch=1, hidden=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            TCNBlock(in_ch, hidden, dilation=1, dropout=dropout),
            TCNBlock(hidden, hidden * 2, dilation=2, dropout=dropout),
            TCNBlock(hidden * 2, hidden * 4, dilation=4, dropout=dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden * 4, num_classes)

    def forward(self, x):
        # x: (B, 1, T)
        x = self.net(x)
        x = self.pool(x).squeeze(-1)  # (B, C)
        return self.fc(x)

In [17]:
def makeDataLoaders(train_df, val_df, test_df, seq_len=2000, batch_size=32):
    train_ds = LightCurveDataset(train_df, seq_len=seq_len)
    val_ds = LightCurveDataset(val_df, seq_len=seq_len)
    test_ds = LightCurveDataset(test_df, seq_len=seq_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [18]:
def train(model, train_loader, val_loader, device=device, num_epochs=20, lr=1e-4, class_weights=None):
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    if class_weights is not None:
        class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * y.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)
                loss = criterion(logits, y)

                val_loss += loss.item() * y.size(0)
                pred = logits.argmax(dim=1)
                correct += (pred == y).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)

        print(f"Epoch {epoch+1:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_tcn.pt")

In [19]:
def splitData(df, test_size=0.2, val_size=0.2, random_state=42):
    train_df, temp_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=df["label"]
    )

    rel_val_size = val_size / (1.0 - test_size)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=rel_val_size,
        random_state=random_state,
        stratify=temp_df["label"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [23]:
# Step 1: Encode labels and split data
df, le = encodeLabels(df)
train_df, val_df, test_df = splitData(df)


In [24]:
# Quick Sanity Check
print("Classes:", list(le.classes_))
print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))
print("Train class counts:", train_df["label"].value_counts().sort_index().to_dict())
print("Val class counts:", val_df["label"].value_counts().sort_index().to_dict())
print("Test class counts:", test_df["label"].value_counts().sort_index().to_dict())


Classes: ['CONFIRMED', 'ECLIPSING BINARY STAR', 'FALSE POSITIVE', 'VARIABLE STAR']
Train / Val / Test sizes: 12418 2328 777
Train class counts: {0: 1578, 1: 2439, 2: 6017, 3: 2384}
Val class counts: {0: 296, 1: 457, 2: 1128, 3: 447}
Test class counts: {0: 98, 1: 153, 2: 377, 3: 149}


In [27]:
# Step 2: Create data loaders
train_loader, val_loader, test_loader = makeDataLoaders(train_df, val_df, test_df)

In [28]:
# Quick Sanity Check

print("\n--- DataLoader sanity check ---")

# Check a single sample
ds = LightCurveDataset(train_df)
x0, y0 = ds[0]

print("Single sample x shape:", x0.shape)   # expect (1, seq_len)
print("Single sample y:", y0)

assert x0.ndim == 2, f"Expected (channels, seq_len), got {x0.shape}"
assert x0.shape[0] == 1, "Expected 1 channel"
assert torch.isfinite(x0).all(), "Non-finite values in single sample"

# Check a batch
x_batch, y_batch = next(iter(train_loader))

print("Batch x shape:", x_batch.shape)      # expect (batch, 1, seq_len)
print("Batch y shape:", y_batch.shape)

assert x_batch.ndim == 3, f"Expected 3D batch, got {x_batch.shape}"
assert x_batch.shape[1] == 1, "Expected channel dimension = 1"
assert y_batch.ndim == 1, "Labels should be 1D"
assert torch.isfinite(x_batch).all(), "Non-finite values in batch"

# Check variation (important!)
print("Batch mean:", x_batch.mean().item())
print("Batch std:", x_batch.std().item())

assert x_batch.std() > 0, "All inputs are constant (bad preprocessing)"

print("DataLoader sanity check passed.\n")


--- DataLoader sanity check ---
Single sample x shape: torch.Size([1, 2000])
Single sample y: tensor(1)
Batch x shape: torch.Size([32, 1, 2000])
Batch y shape: torch.Size([32])
Batch mean: 5.0514934457623895e-09
Batch std: 1.0000077486038208
DataLoader sanity check passed.



In [29]:
# Step 3: Device and Model

device = torch.device("mps")

model = TCNClassifier(num_classes=len(le.classes_), in_ch=1).to(device)

In [ ]:
# Sanity Check 

print("\n--- Model / Device sanity check ---")

print("Device:", device)

param_device = next(model.parameters()).device
print("Model parameters device:", param_device)
assert param_device.type == device.type, "Model is not on the correct device!"

x_batch, y_batch = next(iter(train_loader))

print("Before move -> x device:", x_batch.device)

x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

print("After move -> x device:", x_batch.device)

# Shape checks
assert x_batch.ndim == 3, f"Expected (batch, channels, seq_len), got {x_batch.shape}"
assert x_batch.shape[1] == 1, f"Expected 1 channel, got {x_batch.shape[1]}"

# Label checks
assert y_batch.min() >= 0, "Negative label found"
assert y_batch.max() < len(le.classes_), "Label index out of range"

# Forward pass
with torch.no_grad():
    logits = model(x_batch)

print("Logits shape:", logits.shape)
print("Logits device:", logits.device)

# Output checks
assert logits.shape[0] == x_batch.shape[0]
assert logits.shape[1] == len(le.classes_)
assert logits.device.type == device.type
assert torch.isfinite(logits).all()

print("Logits sample:", logits[0].cpu().numpy())
assert logits.std() > 0, "Model outputs are constant"

# Backward check (important on MPS)
model.train()
logits = model(x_batch[:2])
loss = torch.nn.CrossEntropyLoss()(logits, y_batch[:2])
loss.backward()

for name, p in model.named_parameters():
    if p.grad is not None:
        assert torch.isfinite(p.grad).all(), f"NaNs in gradients of {name}"
        break

model.zero_grad()

print("Model/device sanity check passed.\n")


--- Model / Device sanity check ---
Device: mps
Model parameters device: mps:0
Before move -> x device: cpu
After move -> x device: mps:0
Logits shape: torch.Size([32, 4])
Logits device: mps:0
Logits sample: [-0.6383628   0.22807255 -0.38350615 -0.23390076]
Model/device sanity check passed.



In [37]:
# Step 4: Sanity Check on a Sample Batch

x_batch, y_batch = next(iter(train_loader))

print("Batch x shape:", x_batch.shape)  # expected: (batch, 1, seq_len)
print("Batch y shape:", y_batch.shape)  # expected: (batch,)
print("x dtype:", x_batch.dtype)
print("y dtype:", y_batch.dtype)

assert x_batch.ndim == 3, f"Expected x to be 3D, got {x_batch.shape}"
assert x_batch.shape[1] == 1, f"Expected 1 input channel, got {x_batch.shape[1]}"
assert y_batch.ndim == 1, f"Expected y to be 1D, got {y_batch.shape}"
assert torch.isfinite(x_batch).all(), "Non-finite values found in x_batch"
assert torch.isfinite(y_batch.float()).all(), "Non-finite values found in y_batch"

# Forward-pass check
with torch.no_grad():
    logits = model(x_batch.to(device))

print("Logits shape:", logits.shape)  # expected: (batch, num_classes)
assert logits.shape[0] == x_batch.shape[0], "Batch size mismatch in logits"
assert logits.shape[1] == len(le.classes_), "Number of output classes mismatch"
assert torch.isfinite(logits).all(), "Non-finite values found in model output"

print("Sanity checks passed.")


Batch x shape: torch.Size([32, 1, 2000])
Batch y shape: torch.Size([32])
x dtype: torch.float32
y dtype: torch.int64
Logits shape: torch.Size([32, 4])
Sanity checks passed.


In [40]:
# Step 5: Loss with class weights

counts = Counter(train_df["label"])
total = sum(counts.values())

weights = torch.tensor(
    [total / counts[i] for i in range(len(le.classes_))],
    dtype=torch.float32
).to(device)

print("Class weights:", weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

Class weights: [7.869455  5.091431  2.0638192 5.208893 ]


In [41]:
# Quick Sanity Check on Potential Overfitting

print("\n--- Overfitting sanity check ---")

model.train()

# grab a tiny batch
x_small, y_small = next(iter(train_loader))
x_small = x_small[:8].to(device)
y_small = y_small[:8].to(device)

print("Tiny batch shape:", x_small.shape)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

losses = []

for step in range(50):
    optimizer.zero_grad()

    logits = model(x_small)
    loss = criterion(logits, y_small)

    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if step % 10 == 0:
        preds = logits.argmax(dim=1)
        acc = (preds == y_small).float().mean().item()
        print(f"Step {step:02d} | loss {loss.item():.4f} | acc {acc:.4f}")

# final check
print("Initial loss:", losses[0])
print("Final loss:", losses[-1])

assert losses[-1] < losses[0], "Loss did not decrease"
print("Overfit sanity check passed.\n")



--- Overfitting sanity check ---
Tiny batch shape: torch.Size([8, 1, 2000])
Step 00 | loss 1.4105 | acc 0.1250
Step 10 | loss 0.2484 | acc 0.8750
Step 20 | loss 0.0389 | acc 1.0000
Step 30 | loss 0.0106 | acc 1.0000
Step 40 | loss 0.0047 | acc 1.0000
Initial loss: 1.4105318784713745
Final loss: 0.0028224915731698275
Overfit sanity check passed.



In [43]:
# Step 6: Train the model
model = TCNClassifier(num_classes=len(le.classes_), in_ch=1).to(device)


best_val_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())
num_epochs = 20

for epoch in range(1, num_epochs + 1):
    # ---- train ----
    model.train()
    train_loss = 0.0
    train_preds = []
    train_targets = []

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        assert torch.isfinite(x).all(), "Non-finite values found in training batch x"

        optimizer.zero_grad()
        logits = model(x)

        assert logits.shape[1] == len(le.classes_), "Logits class dimension mismatch"
        assert torch.isfinite(logits).all(), "Non-finite values found in logits"

        loss = criterion(logits, y)
        assert torch.isfinite(loss), "Non-finite loss encountered"

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        train_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
        train_targets.extend(y.detach().cpu().numpy())

    train_loss /= len(train_loader.dataset)
    train_acc = accuracy_score(train_targets, train_preds)
    train_f1 = f1_score(train_targets, train_preds, average="macro")

    # ---- validate ----
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            assert torch.isfinite(x).all(), "Non-finite values found in validation batch x"

            logits = model(x)
            assert torch.isfinite(logits).all(), "Non-finite values found in validation logits"

            loss = criterion(logits, y)
            assert torch.isfinite(loss), "Non-finite validation loss encountered"

            val_loss += loss.item() * x.size(0)
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
            val_targets.extend(y.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro")

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "best_tcn.pt")
        print("  saved best_tcn.pt")

KeyboardInterrupt: 

In [ ]:
# Step 7: Load best model and evaluate on test set

model.load_state_dict(best_state)
model.eval()

In [ ]:
# Step 8: Evaluate on test set

test_preds = []
test_targets = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)

        assert torch.isfinite(logits).all(), "Non-finite values found in test logits"

        test_preds.extend(logits.argmax(dim=1).cpu().numpy())
        test_targets.extend(y.numpy())

test_acc = accuracy_score(test_targets, test_preds)
test_f1 = f1_score(test_targets, test_preds, average="macro")

print("\nTest accuracy:", round(test_acc, 4))
print("Test macro F1:", round(test_f1, 4))
print("\nClassification report:\n")
print(classification_report(test_targets, test_preds, target_names=le.classes_))
print("Confusion matrix:\n")
print(confusion_matrix(test_targets, test_preds))